<a href="https://colab.research.google.com/github/NadiaTeta/group1/blob/main/01_tokens_and_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

torch.manual_seed(42)   # so random results are the same for everyone
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the already-trained GPT-2 (small version) from Hugging Face
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained(
    "gpt2",
    output_attentions=True,         # lets us look at attention weights
    output_hidden_states=True,      # lets us look at the numbers after each layer
    attn_implementation="eager",    # needed so attention weights can be returned
).to(device)
model.eval()                        # we're exploring, not training

print("Device:", device)
print(f"GPT-2 loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} million parameters")

# shared example sentences: the same meaning in English and Kinyarwanda
english = "The people of Kicukiro say they have not had water for three days."
kinyarwanda = "Abaturage bo mu Kicukiro bavuga ko bamaze iminsi itatu nta mazi bafite."

for name, sentence in [("English", english), ("Kinyarwanda", kinyarwanda)]:
    tokens = tokenizer.tokenize(sentence)
    print(f"\n{name}: {len(tokens)} tokens")
    print(tokens)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device: cpu
GPT-2 loaded: 124 million parameters

English: 17 tokens
['The', 'Ġpeople', 'Ġof', 'ĠK', 'ic', 'uk', 'iro', 'Ġsay', 'Ġthey', 'Ġhave', 'Ġnot', 'Ġhad', 'Ġwater', 'Ġfor', 'Ġthree', 'Ġdays', '.']

Kinyarwanda: 29 tokens
['Ab', 'atur', 'age', 'Ġbo', 'Ġmu', 'ĠK', 'ic', 'uk', 'iro', 'Ġb', 'av', 'uga', 'Ġko', 'Ġb', 'am', 'aze', 'Ġim', 'ins', 'i', 'Ġit', 'atu', 'Ġn', 'ta', 'Ġm', 'azi', 'Ġb', 'af', 'ite', '.']


In [5]:
import pandas as pd

rows = []
for name, s in [("English", english), ("Kinyarwanda", kinyarwanda)]:
    n_tokens = len(tokenizer.encode(s))
    n_words = len(s.split())
    rows.append({
        "language": name,
        "words": n_words,
        "tokens": n_tokens,
        "tokens per word": round(n_tokens / n_words, 2),
    })

display(pd.DataFrame(rows))
print(f"Kinyarwanda needs {rows[1]['tokens'] / rows[0]['tokens']:.2f}x as many tokens as English.")

# the same word in both languages, with a leading space as in running text
print("\nSingle-word splits:")
for en, rw in [("water", "amazi"), ("three", "itatu"), ("days", "iminsi"), ("say", "bavuga")]:
    print(f"{en:>8}: {tokenizer.tokenize(' ' + en)}")
    print(f"{rw:>8}: {tokenizer.tokenize(' ' + rw)}")

,language,words,tokens,tokens per word
0,English,13,17,1.31
1,Kinyarwanda,12,29,2.42


Kinyarwanda needs 1.71x as many tokens as English.

Single-word splits:
   water: ['Ġwater']
   amazi: ['Ġam', 'azi']
   three: ['Ġthree']
   itatu: ['Ġit', 'atu']
    days: ['Ġdays']
  iminsi: ['Ġim', 'ins', 'i']
     say: ['Ġsay']
  bavuga: ['Ġb', 'av', 'uga']


In [6]:
def generate(prompt, temperature, max_new_tokens=40, seed=42):
    torch.manual_seed(seed)  # same seed per call, so only temperature differs
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=True,
            temperature=temperature,
            top_k=0,          # turn off top-k and top-p so temperature is the only change
            top_p=1.0,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,  # GPT-2 has no pad token
            output_attentions=False,       # your model sets these to True by default;
            output_hidden_states=False,    # turn them off here to save memory
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompts = [
    "The people of Kicukiro say",
    "In the future, schools in Rwanda will",
    "Abaturage bo mu Kicukiro bavuga ko",
]

for p in prompts:
    print("\n" + "=" * 70)
    print("PROMPT:", p)
    for t in [0.2, 0.7, 1.0, 1.5]:
        print(f"\n--- temperature {t} ---")
        print(generate(p, t))


PROMPT: The people of Kicukiro say

--- temperature 0.2 ---
The people of Kicukiro say that they are not afraid of the people of Kicukiro. They are afraid of the people of Kicukiro. They are afraid of the people of Kicukiro. They are

--- temperature 0.7 ---
The people of Kicukiro say that they've been working on a plan for the past few years to make a new city, and it's a lot of work. It's not an easy project to get started. It takes a

--- temperature 1.0 ---
The people of Kicukiro say that there was a furnace at Intoreo, but it is not like there was a fire, it just fueled. I believe that AC power was used, Salwaelvedr made gas heat

--- temperature 1.5 ---
The people of Kicukiro say Juno bound objects to moves at 92 miles per hour (152 km per year), weighed would we say it misses Yamato 2004)." Takeshi Ito – Thomas Colombat Sal Meielvedrone Park Alexandria

PROMPT: In the future, schools in Rwanda will

--- temperature 0.2 ---
In the future, schools in Rwanda will be able to offer 